# Synthetic SMS Spam Dataset Generation using GPT2

In [9]:
# Install Transformers library
!pip install transformers

In [10]:
import torch
from transformers import GPT2LMHeadModel, GPT2Tokenizer
import random

In [11]:
# Load pre-trained GPT-2 and tokenizer
model_name = "gpt2"  # Use "distilgpt2" for faster but smaller model

tokenizer = GPT2Tokenizer.from_pretrained(model_name)
model = GPT2LMHeadModel.from_pretrained(model_name)
model.eval()

# Move model to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)


GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)

In [12]:
# Prompts for generating spam and ham messages
spam_prompts = [
    "Click on the link below to win",
    "Store special sale offer item for 2$",
    "You won a prize in our lucky draw",
    "Exclusive offer just for you",
    "Claim your free gift now"
]

ham_prompts = [
    "Hi bro how are you doing today",
    "See you later, thanks for the help with homework",
    "Will you come online to play games",
    "Let's meet up for lunch",
    "Hey, what are your weekend plans?"
]


In [13]:
# Function to generate SMS messages
def generate_sms(prompt, max_length=40):
    inputs = tokenizer.encode(prompt, return_tensors="pt").to(device)
    outputs = model.generate(
        inputs,
        max_length=max_length,
        num_return_sequences=1,
        do_sample=True,
        top_k=50,
        top_p=0.95,
        temperature=0.9,
        pad_token_id=tokenizer.eos_token_id
    )
    text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return text.replace(prompt, "").strip().split('\n')[0]


In [6]:
# Generate 2000 spam and 2000 ham messages
synthetic_data = []

for _ in range(2000):
    prompt = random.choice(spam_prompts)
    msg = generate_sms(prompt)
    synthetic_data.append(("spam", msg))

for _ in range(2000):
    prompt = random.choice(ham_prompts)
    msg = generate_sms(prompt)
    synthetic_data.append(("ham", msg))

# Shuffle the dataset
random.shuffle(synthetic_data)


The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


In [7]:
import csv

# Save as CSV, but using tab delimiter and no header
with open("GPT2_Synthetic_SMSSpamCollection.csv", "w", encoding="utf-8", newline='') as f:
    writer = csv.writer(f, delimiter='\t', quoting=csv.QUOTE_MINIMAL)
    for label, message in synthetic_data:
        clean_message = message.replace('\t', ' ').replace('\n', ' ').strip()
        writer.writerow([label, clean_message])

print("Synthetic SMS data saved to 'GPT2_Synthetic_SMSSpamCollection.csv'")


Synthetic SMS data saved to 'GPT2_Synthetic_SMSSpamCollection.csv'


In [15]:
import pandas as pd


# Load the synthetic SMS data from a tab-separated CSV (no header)
synthetic_df = pd.read_csv("GPT2_Synthetic_SMSSpamCollection.csv", sep="\t", header=None, names=["label", "message"])

# Display the first few rows
display(synthetic_df.head())

,label,message
0,ham,???! You are my best friend! I want you to be ...
1,ham,.
2,ham,What do you do with your time?
3,spam,!
4,ham,". I'm looking for something, i mean there's a ..."


In [19]:
import pandas as pd

# Load the synthetic dataset (tab-separated, no header)
synthetic_df = pd.read_csv("GPT2_Synthetic_SMSSpamCollection.csv", sep="\t", header=None, names=["label", "message"])

# Count each label
label_counts = synthetic_df['label'].value_counts()

print("📊 Synthetic Dataset Label Distribution:")
print(label_counts)

📊 Synthetic Dataset Label Distribution:
label
ham     2000
spam    2000
Name: count, dtype: int64
